In [1]:
# %%
from __future__ import annotations

import math
from dataclasses import asdict, dataclass
from typing import Dict, List, Tuple

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


def veilige_del(a: float, b: float, fallback: float = float("nan")) -> float:
    return fallback if abs(b) < 1e-16 else a / b


def mm(waarde_m: float) -> float:
    return 1000.0 * waarde_m


def m_uit_mm(waarde_mm: float) -> float:
    return waarde_mm / 1000.0


def rond_naar_10_mm_boven(waarde_mm: float) -> int:
    return int(math.ceil(waarde_mm / 10.0) * 10.0)


def unieke_reeks_mm(*waarden: int, minimum: int = 0) -> Tuple[int, ...]:
    out = sorted({int(v) for v in waarden if int(v) >= minimum})
    return tuple(out)


def toon_dataframe(naam: str, df: pd.DataFrame, max_rijen: int = 20) -> None:
    print(f"\n=== {naam.upper()} ===")
    if len(df) > max_rijen:
        print(df.head(max_rijen).to_string(index=False))
        print(f"... ({len(df)} rijen in totaal)")
    else:
        print(df.to_string(index=False))


try:
    resultaten_interface
except NameError:
    resultaten_interface = {
        "export_voor_volgende_modules": {
            "Fz_max_N": 1289.2482239760645,
            "Mx_max_Nm": 40.95,  # Tom v19.0443845,
            "My_max_Nm": 493.68779791405746,
            "Mz_max_Nm": 1.38996,
        }
    }

try:
    resultaten_geleidingen
except NameError:
    resultaten_geleidingen = {
        "export_voor_volgende_modules": {
            "F_blok_max_N": 1041.968456,
            "railafstand_m": 0.22,
            "blokafstand_m": 0.35,
        }
    }

try:
    resultaten_spindel
except NameError:
    resultaten_spindel = {
        "export_voor_volgende_modules": {
            "nominale_diameter_spindel_m": 0.040,
            "m1_schacht_totaal_m": 2.478,
            "m1_slag_m": 2.200,
        }
    }

try:
    resultaten_motor
except NameError:
    resultaten_motor = {
        "export_voor_volgende_modules": {
            "motor_flange_mm": 130.0,
            "motor_body_diameter_mm": 130.0,
            "motor_body_length_mm": 180.0,
            "motor_shaft_diameter_mm": 19.0,
        }
    }

try:
    resultaten_koppeling_notebook
except NameError:
    resultaten_koppeling_notebook = {
        "configuratie_koppeling": {
            "diameter_motorkant_m": 0.019,
            "diameter_lastkant_m": 0.024,
        },
        "export_voor_volgende_modules": {"d_min_torsie_mm": 8.0},
    }

try:
    resultaten_montageplaat
except NameError:
    resultaten_montageplaat = {
        "export_voor_volgende_modules": {
            "voorkeursplaat_haalbaar": True,
            "voorkeursplaat_breedte_mm": 440.0,
            "voorkeursplaat_hoogte_mm": 360.0,
            "voorkeursplaat_dikte_mm": 20.0,
            "plaat_keepout_breedte_mm": 170.0,
            "plaat_keepout_hoogte_mm": 150.0,
        }
    }


@dataclass(frozen=True)
class Materiaal:
    naam: str
    E_Pa: float
    rho_kg_m3: float
    sigma_vloei_Pa: float
    nu: float = 0.30

    @property
    def G_Pa(self) -> float:
        return self.E_Pa / (2.0 * (1.0 + self.nu))


MATERIALEN: Dict[str, Materiaal] = {
    "staal_s355": Materiaal("Staal S355", 210e9, 7850.0, 355e6, 0.30),
    "aluminium_6082_t6": Materiaal("Aluminium 6082-T6", 70e9, 2700.0, 260e6, 0.33),
}


@dataclass(frozen=True)
class Plaatconcept:
    materiaal: str
    breedte_m: float
    hoogte_m: float
    dikte_m: float
    aantal_ribben: int
    hoogte_rib_m: float
    dikte_rib_m: float


@dataclass(frozen=True)
class FrameConcept:
    materiaal: str
    breedte_buiten_m: float
    hoogte_buiten_m: float
    wanddikte_m: float


@dataclass(frozen=True)
class BehuizingConcept:
    ondersteuningsplaat: Plaatconcept
    dikte_zijplaat_m: float
    dikte_voorplaat_m: float
    dikte_achterplaat_m: float


@dataclass(frozen=True)
class ConfiguratieM2:
    delta_theta_rad: float = math.pi  # Tom v19
    t_max_s: float = 2.0
    rho_aluminium_kg_m3: float = 2700.0
    rho_staal_kg_m3: float = 7850.0
    hoogte_uitgaandearm_m: float = 0.10
    breedte_uitgaandearm_m: float = 0.10
    lengte_uitgaandearm_x2_m: float = 0.46
    lengte_uitgaandearm_x1_m: float = 0.20
    massa_last_kg: float = 8.0
    t_dikte_balk_m: float = 0.015  # Tom v19: wanddikte arm bijgewerkt
    lengte_uitgaandeas1_m: float = 0.20
    lengte_uitgaandeas2_m: float = 0.20
    lengte_uitgaandeas3_m: float = 0.20
    d_uitgaandeas_gekozen_m: float = 0.030
    theta_limiet_rad: float = 0.00436
    K_A_tandriem: float = 1.5
    steek_tandriem_m: float = 5e-3
    aantal_tanden_schijf: int = 150
    definitief_aantal_tanden_riem: int = 215
    riembreedte_m: float = 16e-3
    voorlopige_asafstand_tandriem_m: float = 0.40
    aantal_tandriemschijven: int = 2
    lengte_spline_as_m: float = 3.5
    lagerafstand_m: float = 0.60
    lagerhoogte_m: float = 0.010
    breedte_kokerM1_m: float = 0.10
    speling_zijplaten_m: float = 0.005
    extra_inkeping_m: float = 0.10
    overspanning_steunpunten_m: float = 0.072
    k_groefkogellager_N_m: float = 1.0e6
    k_m1rails_N_m: float = 1.0e5


@dataclass(frozen=True)
class ConfiguratieFrame:
    lengte_frame_m: float
    positie_last_frame_m: float
    veiligheidsfactor_frame: float = 2.0
    toegelaten_doorbuiging_frame_mm: float = 0.25
    toegelaten_rotatie_frame_deg: float = 0.08
    toegelaten_massa_frame_kg: float = 250.0
    breedtes_frame_mm: Tuple[int, ...] = (500, 520, 540)
    hoogtes_frame_mm: Tuple[int, ...] = (420, 440, 460)
    wanddiktes_frame_mm: Tuple[int, ...] = (4, 5, 6, 8, 10)
    voorkeursmateriaal_frame: str = "staal_s355"
    voorkeurs_breedte_frame_mm: float = 520.0
    voorkeurs_hoogte_frame_mm: float = 440.0
    voorkeurs_wanddikte_frame_mm: float = 5.0


@dataclass(frozen=True)
class ConfiguratieBehuizing:
    lengte_ondersteuningsplaat_min_m: float
    breedte_ondersteuningsplaat_min_m: float
    opbouwhoogte_ondersteuningsplaat_m: float
    hoogte_zijplaat_min_m: float
    lengte_zijplaatA_min_m: float
    lengte_zijplaatB_min_m: float
    hoogte_inkeping_min_m: float
    lengte_inkeping_min_m: float
    lengte_voorplaat_min_m: float
    hoogte_voorplaat_min_m: float
    lengte_achterplaat_min_m: float
    hoogte_achterplaat_min_m: float
    overspanning_steunpunten_m: float
    k_groefkogellager_N_m: float
    k_m1rails_N_m: float
    veiligheidsfactor_behuizing: float = 2.0
    toegelaten_doorbuiging_ondersteuningsplaat_mm: float = 0.15
    toegelaten_rotatie_ondersteuningsplaat_graden: float = 0.08
    toegelaten_doorbuiging_paneel_mm: float = 1.00
    toegelaten_massa_behuizing_kg: float = 80.0
    breedtes_ondersteuningsplaat_mm: Tuple[int, ...] = (660, 680, 700)
    hoogtes_ondersteuningsplaat_mm: Tuple[int, ...] = (280, 300, 320)
    diktes_ondersteuningsplaat_mm: Tuple[int, ...] = (8, 10, 12, 15)
    aantallen_ribben: Tuple[int, ...] = (0, 2, 3)
    hoogtes_rib_mm: Tuple[int, ...] = (0, 40, 60, 80)
    diktes_rib_mm: Tuple[int, ...] = (0, 5, 6, 8)
    diktes_zijplaat_mm: Tuple[int, ...] = (4, 5, 6, 8)
    diktes_voorplaat_mm: Tuple[int, ...] = (4, 5, 6, 8)
    diktes_achterplaat_mm: Tuple[int, ...] = (4, 5, 6, 8)
    aantal_topkandidaten_behuizing: int = 15
    voorkeursmateriaal_behuizing: str = "aluminium_6082_t6"
    voorkeurs_lengte_ondersteuningsplaat_mm: float = 700.0
    voorkeurs_breedte_ondersteuningsplaat_mm: float = 300.0
    voorkeurs_dikte_ondersteuningsplaat_mm: float = 12.0
    voorkeurs_aantal_ribben: int = 2
    voorkeurs_hoogte_rib_mm: float = 60.0
    voorkeurs_dikte_rib_mm: float = 6.0
    voorkeurs_dikte_zijplaat_mm: float = 6.0
    voorkeurs_dikte_voorplaat_mm: float = 5.0
    voorkeurs_dikte_achterplaat_mm: float = 5.0


def maak_interface_lasten_frame() -> Dict[str, float]:
    exp = resultaten_interface["export_voor_volgende_modules"]
    out = {
        "Fz_N": float(exp["Fz_max_N"]),
        "Mx_Nm": float(exp["Mx_max_Nm"]),
        "My_Nm": float(exp["My_max_Nm"]),
        "Mz_Nm": float(exp["Mz_max_Nm"]),
    }
    try:
        out["F_blok_max_N"] = float(resultaten_geleidingen["export_voor_volgende_modules"]["F_blok_max_N"])
    except Exception:
        out["F_blok_max_N"] = 0.0
    return out


def maak_m2_config() -> ConfiguratieM2:
    return ConfiguratieM2()


def bereken_m2_kengetallen(cfg: ConfiguratieM2) -> Dict[str, float]:
    alpha_max = 4.5 * cfg.delta_theta_rad / (cfg.t_max_s ** 2)
    alpha_gem = cfg.delta_theta_rad / (cfg.t_max_s ** 2)
    omega_max = 1.5 * cfg.delta_theta_rad / cfg.t_max_s
    omega_gem = cfg.delta_theta_rad / cfg.t_max_s

    A_hol = (
        cfg.breedte_uitgaandearm_m * cfg.hoogte_uitgaandearm_m
        - (cfg.breedte_uitgaandearm_m - 2.0 * cfg.t_dikte_balk_m)
        * (cfg.hoogte_uitgaandearm_m - 2.0 * cfg.t_dikte_balk_m)
    )
    volume_arm = A_hol * (cfg.lengte_uitgaandearm_x2_m + cfg.lengte_uitgaandearm_x1_m)
    massa_arm = volume_arm * cfg.rho_aluminium_kg_m3
    I_arm = (
        cfg.breedte_uitgaandearm_m * cfg.hoogte_uitgaandearm_m ** 3
        - (cfg.breedte_uitgaandearm_m - 2.0 * cfg.t_dikte_balk_m)
        * (cfg.hoogte_uitgaandearm_m - 2.0 * cfg.t_dikte_balk_m) ** 3
    ) / 12.0
    W_arm = veilige_del(I_arm, cfg.hoogte_uitgaandearm_m / 2.0, float("nan"))
    J_arm = (
        (1.0 / 3.0)
        * (massa_arm / (cfg.lengte_uitgaandearm_x2_m + cfg.lengte_uitgaandearm_x1_m))
        * (cfg.lengte_uitgaandearm_x2_m ** 3 + cfg.lengte_uitgaandearm_x1_m ** 3)
    )
    J_last = cfg.massa_last_kg * cfg.lengte_uitgaandearm_x2_m ** 2
    J_totaal = J_arm + J_last

    T_dynamisch = J_totaal * alpha_max
    g = 9.81
    M_statisch = (
        cfg.massa_last_kg * g * cfg.lengte_uitgaandearm_x2_m
        + massa_arm * g * ((cfg.lengte_uitgaandearm_x2_m - cfg.lengte_uitgaandearm_x1_m) / 2.0)
    )
    M_totaal = math.sqrt(T_dynamisch ** 2 + M_statisch ** 2)
    sigma_arm = veilige_del(M_totaal, W_arm, float("nan"))
    P_motor = T_dynamisch * omega_max

    n_uitgaandeas_rpm = omega_max * 60.0 / (2.0 * math.pi)
    lengte_uitgaandeas = cfg.lengte_uitgaandeas1_m + cfg.lengte_uitgaandeas2_m + cfg.lengte_uitgaandeas3_m
    I_uitgaandeas = math.pi * (cfg.d_uitgaandeas_gekozen_m / 2.0) ** 4 / 4.0
    M_max_doorbuiging = (
        cfg.theta_limiet_rad * 3.0 * 210e9 * I_uitgaandeas / lengte_uitgaandeas
    )

    T_tandriem = cfg.K_A_tandriem * T_dynamisch
    steekcirkeldiameter = cfg.aantal_tanden_schijf * cfg.steek_tandriem_m / math.pi
    def_lengte_tandriem = cfg.definitief_aantal_tanden_riem * cfg.steek_tandriem_m
    def_asafstand_tandriem = (
        def_lengte_tandriem / 4.0
        - (math.pi / 8.0) * (2.0 * steekcirkeldiameter)
        + math.sqrt(
            (def_lengte_tandriem / 4.0 - (math.pi / 8.0) * (2.0 * steekcirkeldiameter)) ** 2
            - 0.0
        )
    )
    snelheid_tandriem = steekcirkeldiameter * math.pi * n_uitgaandeas_rpm / 60.0
    P_tandriem = T_tandriem * omega_max
    omtrekskracht_tandriem = veilige_del(P_tandriem, snelheid_tandriem, float("nan"))
    askracht_tandriem = 1.1 * omtrekskracht_tandriem
    omspanningshoek_graden = 180.0
    tanden_in_ingreep = omspanningshoek_graden / 360.0 * cfg.definitief_aantal_tanden_riem

    massa_tandriemschijf = (
        math.pi * (steekcirkeldiameter / 2.0) ** 2 * cfg.riembreedte_m * cfg.rho_aluminium_kg_m3
    )
    totale_massa_tandriemschijven = cfg.aantal_tandriemschijven * massa_tandriemschijf

    I_spline = math.pi * (cfg.d_uitgaandeas_gekozen_m / 2.0) ** 4 / 4.0
    k_as = 48.0 * 210e9 * I_spline / (cfg.lengte_spline_as_m ** 3)

    return {
        "alpha_max_rad_s2": alpha_max,
        "alpha_gem_rad_s2": alpha_gem,
        "omega_max_rad_s": omega_max,
        "omega_gem_rad_s": omega_gem,
        "massa_uitgaandearm_kg": massa_arm,
        "I_uitgaandearm_m4": I_arm,
        "J_totaal_kgm2": J_totaal,
        "T_dynamisch_Nm": T_dynamisch,
        "M_statisch_Nm": M_statisch,
        "M_totaal_Nm": M_totaal,
        "sigma_arm_Pa": sigma_arm,
        "P_motor_W": P_motor,
        "n_uitgaandeas_rpm": n_uitgaandeas_rpm,
        "lengte_uitgaandeas_m": lengte_uitgaandeas,
        "M_max_doorbuiging_as_Nm": M_max_doorbuiging,
        "T_tandriem_Nm": T_tandriem,
        "steekcirkeldiameter_tandriem_m": steekcirkeldiameter,
        "definitieve_lengte_tandriem_m": def_lengte_tandriem,
        "definitieve_asafstand_tandriem_m": def_asafstand_tandriem,
        "snelheid_tandriem_m_s": snelheid_tandriem,
        "omtrekskracht_tandriem_N": omtrekskracht_tandriem,
        "askracht_tandriem_N": askracht_tandriem,
        "tanden_in_ingreep": tanden_in_ingreep,
        "massa_tandriemschijf_kg": massa_tandriemschijf,
        "totale_massa_tandriemschijven_kg": totale_massa_tandriemschijven,
        "k_as_N_m": k_as,
    }


def maak_afgeleide_frameconfig() -> ConfiguratieFrame:
    sp = resultaten_spindel["export_voor_volgende_modules"]
    mot = resultaten_motor["export_voor_volgende_modules"]
    plaat = resultaten_montageplaat["export_voor_volgende_modules"]
    kop_cfg = resultaten_koppeling_notebook.get("configuratie_koppeling", {})

    shaft_total_mm = 1000.0 * float(sp.get("m1_schacht_totaal_m", 2.478))
    motor_len_mm = float(mot.get("motor_body_length_mm", 180.0))
    motor_flange_mm = float(mot.get("motor_flange_mm", 130.0))
    motor_diameter_mm = float(mot.get("motor_body_diameter_mm", motor_flange_mm))
    plate_w_mm = float(plaat.get("voorkeursplaat_breedte_mm", 440.0))
    plate_h_mm = float(plaat.get("voorkeursplaat_hoogte_mm", 360.0))
    keepout_w_mm = float(plaat.get("plaat_keepout_breedte_mm", 170.0))
    keepout_h_mm = float(plaat.get("plaat_keepout_hoogte_mm", 150.0))
    coupling_len_mm = max(
        90.0,
        3.5 * 1000.0 * float(kop_cfg.get("diameter_lastkant_m", 0.024)),
    )

    lengte_frame_mm = shaft_total_mm + motor_len_mm + coupling_len_mm + 150.0
    req_breedte_mm = max(plate_w_mm + 80.0, motor_flange_mm + 180.0, keepout_w_mm + 300.0)
    req_hoogte_mm = max(plate_h_mm + 80.0, motor_diameter_mm + 180.0, keepout_h_mm + 280.0)
    voorkeurs_breedte_mm = rond_naar_10_mm_boven(req_breedte_mm)
    voorkeurs_hoogte_mm = rond_naar_10_mm_boven(req_hoogte_mm)

    breedtes = unieke_reeks_mm(
        voorkeurs_breedte_mm - 20,
        voorkeurs_breedte_mm,
        voorkeurs_breedte_mm + 20,
        minimum=420,
    )
    hoogtes = unieke_reeks_mm(
        voorkeurs_hoogte_mm - 20,
        voorkeurs_hoogte_mm,
        voorkeurs_hoogte_mm + 20,
        minimum=360,
    )

    return ConfiguratieFrame(
        lengte_frame_m=m_uit_mm(lengte_frame_mm),
        positie_last_frame_m=m_uit_mm(lengte_frame_mm) / 2.0,
        breedtes_frame_mm=breedtes,
        hoogtes_frame_mm=hoogtes,
        voorkeurs_breedte_frame_mm=voorkeurs_breedte_mm,
        voorkeurs_hoogte_frame_mm=voorkeurs_hoogte_mm,
    )


def maak_afgeleide_behuizingconfig(cfg_m2: ConfiguratieM2, m2: Dict[str, float]) -> ConfiguratieBehuizing:
    lengte_support_mm_req = rond_naar_10_mm_boven(
        1000.0 * max(
            0.70,
            m2["definitieve_asafstand_tandriem_m"] + 0.50,
            cfg_m2.lengte_uitgaandearm_x1_m + cfg_m2.extra_inkeping_m + 0.35,
        )
    )
    breedte_support_mm_req = rond_naar_10_mm_boven(1000.0 * max(0.30, cfg_m2.breedte_kokerM1_m + 0.20))
    opbouwhoogte_mm = 100.0
    hoogte_zijplaat_mm_req = rond_naar_10_mm_boven(
        1000.0 * (cfg_m2.lagerafstand_m + 2.0 * cfg_m2.lagerhoogte_m + 2.0 * opbouwhoogte_mm / 1000.0)
    )
    lengte_zijplaatA_mm_req = rond_naar_10_mm_boven(1000.0 * (cfg_m2.breedte_kokerM1_m + cfg_m2.speling_zijplaten_m))
    lengte_zijplaatB_mm_req = max(200, lengte_support_mm_req - lengte_zijplaatA_mm_req)
    hoogte_inkeping_mm_req = rond_naar_10_mm_boven(
        1000.0 * (cfg_m2.hoogte_uitgaandearm_m + 2.0 * cfg_m2.speling_zijplaten_m)
    )
    lengte_inkeping_mm_req = rond_naar_10_mm_boven(
        1000.0 * (cfg_m2.lengte_uitgaandearm_x1_m + cfg_m2.extra_inkeping_m + cfg_m2.speling_zijplaten_m)
    )
    hoogte_voorplaat_mm_req = max(200, hoogte_zijplaat_mm_req - hoogte_inkeping_mm_req)

    support_lengths = unieke_reeks_mm(
        lengte_support_mm_req - 20,
        lengte_support_mm_req,
        lengte_support_mm_req + 20,
        minimum=620,
    )
    support_widths = unieke_reeks_mm(
        breedte_support_mm_req - 20,
        breedte_support_mm_req,
        breedte_support_mm_req + 20,
        minimum=260,
    )

    return ConfiguratieBehuizing(
        lengte_ondersteuningsplaat_min_m=m_uit_mm(lengte_support_mm_req),
        breedte_ondersteuningsplaat_min_m=m_uit_mm(breedte_support_mm_req),
        opbouwhoogte_ondersteuningsplaat_m=m_uit_mm(opbouwhoogte_mm),
        hoogte_zijplaat_min_m=m_uit_mm(hoogte_zijplaat_mm_req),
        lengte_zijplaatA_min_m=m_uit_mm(lengte_zijplaatA_mm_req),
        lengte_zijplaatB_min_m=m_uit_mm(lengte_zijplaatB_mm_req),
        hoogte_inkeping_min_m=m_uit_mm(hoogte_inkeping_mm_req),
        lengte_inkeping_min_m=m_uit_mm(lengte_inkeping_mm_req),
        lengte_voorplaat_min_m=m_uit_mm(breedte_support_mm_req),
        hoogte_voorplaat_min_m=m_uit_mm(hoogte_voorplaat_mm_req),
        lengte_achterplaat_min_m=m_uit_mm(breedte_support_mm_req),
        hoogte_achterplaat_min_m=m_uit_mm(hoogte_zijplaat_mm_req),
        overspanning_steunpunten_m=cfg_m2.overspanning_steunpunten_m,
        k_groefkogellager_N_m=cfg_m2.k_groefkogellager_N_m,
        k_m1rails_N_m=cfg_m2.k_m1rails_N_m,
        breedtes_ondersteuningsplaat_mm=support_lengths,
        hoogtes_ondersteuningsplaat_mm=support_widths,
        voorkeurs_lengte_ondersteuningsplaat_mm=lengte_support_mm_req,
        voorkeurs_breedte_ondersteuningsplaat_mm=breedte_support_mm_req,
    )


def doorbuiging_eenvoudig_ondersteund_puntlast(P_N: float, L_m: float, x_m: float, E_Pa: float, I_m4: float) -> float:
    if min(L_m, E_Pa, I_m4) <= 0.0:
        return float("nan")
    a = x_m
    b = L_m - x_m
    return P_N * a * b * (L_m ** 2 - a ** 2 - b ** 2) / (6.0 * E_Pa * I_m4 * L_m)


def moment_eenvoudig_ondersteund_puntlast(P_N: float, L_m: float, x_m: float) -> float:
    a = x_m
    b = L_m - x_m
    return (P_N * b / L_m) * a


def rechthoekige_koker_eigenschappen(b_m: float, h_m: float, t_m: float) -> Dict[str, float]:
    if min(b_m, h_m, t_m) <= 0.0 or 2.0 * t_m >= min(b_m, h_m):
        return {
            "A_m2": float("nan"),
            "Ix_m4": float("nan"),
            "Iy_m4": float("nan"),
            "Wx_m3": float("nan"),
            "Wy_m3": float("nan"),
            "J_t_m4": float("nan"),
        }
    b_in = b_m - 2.0 * t_m
    h_in = h_m - 2.0 * t_m
    A = b_m * h_m - b_in * h_in
    Ix = (b_m * h_m ** 3 - b_in * h_in ** 3) / 12.0
    Iy = (h_m * b_m ** 3 - h_in * b_in ** 3) / 12.0
    Wx = veilige_del(Ix, h_m / 2.0, float("nan"))
    Wy = veilige_del(Iy, b_m / 2.0, float("nan"))
    b_mid = b_m - t_m
    h_mid = h_m - t_m
    A_mid = b_mid * h_mid
    J_t = 4.0 * A_mid ** 2 / (2.0 * (b_mid / t_m + h_mid / t_m))
    return {"A_m2": A, "Ix_m4": Ix, "Iy_m4": Iy, "Wx_m3": Wx, "Wy_m3": Wy, "J_t_m4": J_t}


def sectie_eigenschappen_plaat(concept: Plaatconcept) -> Dict[str, float]:
    materiaal = MATERIALEN[concept.materiaal]
    b = concept.breedte_m
    h = concept.hoogte_m
    t = concept.dikte_m
    I_plaat_x = t * h ** 3 / 12.0
    I_plaat_y = h * t ** 3 / 12.0
    I_ribben_x = 0.0
    I_ribben_y = 0.0
    A_ribben = 0.0
    if concept.aantal_ribben > 0 and concept.hoogte_rib_m > 0 and concept.dikte_rib_m > 0:
        A_rib = concept.dikte_rib_m * concept.hoogte_rib_m
        I_rib_lokaal_x = concept.dikte_rib_m * concept.hoogte_rib_m ** 3 / 12.0
        I_rib_lokaal_y = concept.hoogte_rib_m * concept.dikte_rib_m ** 3 / 12.0
        afstand_neutraal = t / 2.0 + concept.hoogte_rib_m / 2.0
        x_posities = np.linspace(-0.35 * b, 0.35 * b, concept.aantal_ribben)
        for x_i in x_posities:
            I_ribben_x += I_rib_lokaal_x + A_rib * afstand_neutraal ** 2
            I_ribben_y += I_rib_lokaal_y + A_rib * (x_i ** 2 + afstand_neutraal ** 2)
        A_ribben = concept.aantal_ribben * A_rib
    I_eq_x = I_plaat_x + I_ribben_x
    I_eq_y = I_plaat_y + I_ribben_y
    W_eq_x = veilige_del(I_eq_x, h / 2.0, float("nan"))
    W_eq_y = veilige_del(I_eq_y, max(t / 2.0, 1e-9), float("nan"))
    massa_plaat = b * h * t * materiaal.rho_kg_m3
    massa_ribben = A_ribben * b * materiaal.rho_kg_m3 if concept.aantal_ribben > 0 else 0.0
    return {
        "I_eq_x_m4": I_eq_x,
        "I_eq_y_m4": I_eq_y,
        "W_eq_x_m3": W_eq_x,
        "W_eq_y_m3": W_eq_y,
        "massa_totaal_kg": massa_plaat + massa_ribben,
    }


def evalueer_frameconcept(concept: FrameConcept, cfg: ConfiguratieFrame, interface_lasten: Dict[str, float]) -> Dict[str, object]:
    mat = MATERIALEN[concept.materiaal]
    props = rechthoekige_koker_eigenschappen(concept.breedte_buiten_m, concept.hoogte_buiten_m, concept.wanddikte_m)
    redenen: List[str] = []
    haalbaar = True
    L = cfg.lengte_frame_m
    x = cfg.positie_last_frame_m
    Mx = abs(interface_lasten["Mx_Nm"])
    My = abs(interface_lasten["My_Nm"])
    Mz = abs(interface_lasten["Mz_Nm"])
    Fz = abs(interface_lasten["Fz_N"])
    sigma_buig = veilige_del(Mx, props["Wx_m3"], 0.0) + veilige_del(My, props["Wy_m3"], 0.0)
    sigma_axiaal = veilige_del(Fz, props["A_m2"], 0.0)
    sigma_eq = sigma_buig + sigma_axiaal
    sigma_toeg = mat.sigma_vloei_Pa / cfg.veiligheidsfactor_frame
    delta = doorbuiging_eenvoudig_ondersteund_puntlast(Fz, L, x, mat.E_Pa, props["Ix_m4"])
    theta_buig = veilige_del(My * L, mat.E_Pa * props["Iy_m4"], 0.0) * 180.0 / math.pi
    theta_torsie = veilige_del(Mz * L, mat.G_Pa * props["J_t_m4"], 0.0) * 180.0 / math.pi
    theta = theta_buig + theta_torsie
    massa = props["A_m2"] * L * mat.rho_kg_m3
    if sigma_eq > sigma_toeg:
        haalbaar = False
        redenen.append("frame spanning te hoog")
    if 1000.0 * delta > cfg.toegelaten_doorbuiging_frame_mm:
        haalbaar = False
        redenen.append("frame doorbuiging te hoog")
    if theta > cfg.toegelaten_rotatie_frame_deg:
        haalbaar = False
        redenen.append("frame rotatie te hoog")
    if massa > cfg.toegelaten_massa_frame_kg:
        haalbaar = False
        redenen.append("frame massa te hoog")
    doel = (
        veilige_del(sigma_eq, sigma_toeg, float("nan")) * 5000.0
        + veilige_del(1000.0 * delta, cfg.toegelaten_doorbuiging_frame_mm, float("nan")) * 4000.0
        + veilige_del(theta, cfg.toegelaten_rotatie_frame_deg, float("nan")) * 3000.0
        + veilige_del(massa, cfg.toegelaten_massa_frame_kg, float("nan"))
        + (0.0 if haalbaar else 1e6)
    )
    return {
        "haalbaar": haalbaar,
        "redenen": redenen,
        "doelfunctie": doel,
        "materiaal": concept.materiaal,
        "breedte_mm": mm(concept.breedte_buiten_m),
        "hoogte_mm": mm(concept.hoogte_buiten_m),
        "wanddikte_mm": mm(concept.wanddikte_m),
        "A_m2": props["A_m2"],
        "Ix_m4": props["Ix_m4"],
        "Iy_m4": props["Iy_m4"],
        "J_t_m4": props["J_t_m4"],
        "sigma_eq_Pa": sigma_eq,
        "sigma_toeg_Pa": sigma_toeg,
        "doorbuiging_mm": 1000.0 * delta,
        "rotatie_deg": theta,
        "massa_kg": massa,
        "benutting_spanning": veilige_del(sigma_eq, sigma_toeg, float("nan")),
        "benutting_doorbuiging": veilige_del(1000.0 * delta, cfg.toegelaten_doorbuiging_frame_mm, float("nan")),
        "benutting_rotatie": veilige_del(theta, cfg.toegelaten_rotatie_frame_deg, float("nan")),
        "benutting_massa": veilige_del(massa, cfg.toegelaten_massa_frame_kg, float("nan")),
    }


def optimaliseer_frame(cfg: ConfiguratieFrame, interface_lasten: Dict[str, float]) -> Tuple[pd.DataFrame, pd.DataFrame]:
    rijen = []
    for materiaal in [cfg.voorkeursmateriaal_frame]:
        for b_mm in cfg.breedtes_frame_mm:
            for h_mm in cfg.hoogtes_frame_mm:
                for t_mm in cfg.wanddiktes_frame_mm:
                    concept = FrameConcept(materiaal, m_uit_mm(b_mm), m_uit_mm(h_mm), m_uit_mm(t_mm))
                    rijen.append(evalueer_frameconcept(concept, cfg, interface_lasten))
    df = pd.DataFrame(rijen).sort_values(["haalbaar", "doelfunctie"], ascending=[False, True]).reset_index(drop=True)
    haalbaar = df[df["haalbaar"]].reset_index(drop=True)
    return df, haalbaar


def maak_voorkeursframe(cfg: ConfiguratieFrame) -> FrameConcept:
    return FrameConcept(
        cfg.voorkeursmateriaal_frame,
        m_uit_mm(cfg.voorkeurs_breedte_frame_mm),
        m_uit_mm(cfg.voorkeurs_hoogte_frame_mm),
        m_uit_mm(cfg.voorkeurs_wanddikte_frame_mm),
    )


def buigstijfheid_ondersteuningsplaat(concept: Plaatconcept, cfg_b: ConfiguratieBehuizing) -> float:
    E = MATERIALEN[concept.materiaal].E_Pa
    I_eq = sectie_eigenschappen_plaat(concept)["I_eq_x_m4"]
    L = cfg_b.overspanning_steunpunten_m
    return veilige_del(48.0 * E * I_eq, L ** 3, float("nan"))


def paneel_strip_respons(
    materiaal: str,
    breedte_m: float,
    hoogte_m: float,
    dikte_m: float,
    puntlast_N: float,
) -> Dict[str, float]:
    mat = MATERIALEN[materiaal]
    if min(breedte_m, hoogte_m, dikte_m) <= 0.0:
        return {"I_m4": float("nan"), "W_m3": float("nan"), "sigma_Pa": float("nan"), "doorbuiging_mm": float("nan"), "massa_kg": float("nan")}
    span = min(breedte_m, hoogte_m)
    breedte_eff = max(breedte_m, hoogte_m)
    I = breedte_eff * dikte_m ** 3 / 12.0
    W = veilige_del(I, dikte_m / 2.0, float("nan"))
    M = puntlast_N * span / 4.0
    sigma = veilige_del(M, W, float("nan"))
    delta = puntlast_N * span ** 3 / (48.0 * mat.E_Pa * I)
    massa = breedte_m * hoogte_m * dikte_m * mat.rho_kg_m3
    return {"I_m4": I, "W_m3": W, "sigma_Pa": sigma, "doorbuiging_mm": 1000.0 * delta, "massa_kg": massa}


def evalueer_behuizingsconcept(
    concept: BehuizingConcept,
    cfg_b: ConfiguratieBehuizing,
    interface_lasten: Dict[str, float],
    m2: Dict[str, float],
) -> Dict[str, object]:
    mat_support = MATERIALEN[concept.ondersteuningsplaat.materiaal]
    redenen: List[str] = []
    haalbaar = True

    if concept.ondersteuningsplaat.breedte_m < cfg_b.lengte_ondersteuningsplaat_min_m:
        haalbaar = False
        redenen.append("ondersteuningsplaat te kort")
    if concept.ondersteuningsplaat.hoogte_m < cfg_b.breedte_ondersteuningsplaat_min_m:
        haalbaar = False
        redenen.append("ondersteuningsplaat te smal")

    k_plaat_single = buigstijfheid_ondersteuningsplaat(concept.ondersteuningsplaat, cfg_b)
    k_plaat_totaal = 2.0 * k_plaat_single
    k_behuizing = 1.0 / (
        veilige_del(1.0, k_plaat_totaal, float("inf"))
        + veilige_del(1.0, cfg_b.k_groefkogellager_N_m, float("inf"))
        + veilige_del(1.0, cfg_b.k_m1rails_N_m, float("inf"))
    )
    verdeling_riemkracht_as = veilige_del(m2["k_as_N_m"], m2["k_as_N_m"] + k_behuizing, 0.0)
    F_behuizing = m2["askracht_tandriem_N"] * (1.0 - verdeling_riemkracht_as)
    P_support = 0.5 * F_behuizing + 0.25 * abs(interface_lasten.get("F_blok_max_N", 0.0))

    props_support = sectie_eigenschappen_plaat(concept.ondersteuningsplaat)
    M_support = P_support * cfg_b.overspanning_steunpunten_m / 4.0
    sigma_support = veilige_del(M_support, props_support["W_eq_x_m3"], 0.0)
    sigma_support_toeg = mat_support.sigma_vloei_Pa / cfg_b.veiligheidsfactor_behuizing
    delta_support = doorbuiging_eenvoudig_ondersteund_puntlast(
        P_support,
        cfg_b.overspanning_steunpunten_m,
        cfg_b.overspanning_steunpunten_m / 2.0,
        mat_support.E_Pa,
        props_support["I_eq_x_m4"],
    )
    theta_support = veilige_del(
        M_support * cfg_b.overspanning_steunpunten_m,
        2.0 * mat_support.E_Pa * props_support["I_eq_x_m4"],
        0.0,
    ) * 180.0 / math.pi

    hoogte_zijplaat = cfg_b.hoogte_zijplaat_min_m
    lengte_zijplaatA = cfg_b.lengte_zijplaatA_min_m
    lengte_zijplaatB = max(cfg_b.lengte_zijplaatB_min_m, concept.ondersteuningsplaat.breedte_m - lengte_zijplaatA)
    hoogte_inkeping = cfg_b.hoogte_inkeping_min_m
    lengte_inkeping = min(cfg_b.lengte_inkeping_min_m, 0.9 * lengte_zijplaatB)
    hoogte_voorplaat = max(cfg_b.hoogte_voorplaat_min_m, hoogte_zijplaat - hoogte_inkeping)

    if hoogte_inkeping >= hoogte_zijplaat:
        haalbaar = False
        redenen.append("inkeping zijplaat te hoog")
    if lengte_inkeping >= lengte_zijplaatB:
        haalbaar = False
        redenen.append("inkeping zijplaat te lang")

    F_panel = max(0.25 * F_behuizing, 0.10 * abs(interface_lasten["Fz_N"]))
    paneel_A = paneel_strip_respons(
        cfg_b.voorkeursmateriaal_behuizing,
        lengte_zijplaatA,
        hoogte_zijplaat,
        concept.dikte_zijplaat_m,
        F_panel,
    )
    paneel_B = paneel_strip_respons(
        cfg_b.voorkeursmateriaal_behuizing,
        lengte_zijplaatB,
        hoogte_zijplaat,
        concept.dikte_zijplaat_m,
        F_panel,
    )
    voorplaat = paneel_strip_respons(
        cfg_b.voorkeursmateriaal_behuizing,
        concept.ondersteuningsplaat.hoogte_m,
        hoogte_voorplaat,
        concept.dikte_voorplaat_m,
        F_panel,
    )
    achterplaat = paneel_strip_respons(
        cfg_b.voorkeursmateriaal_behuizing,
        concept.ondersteuningsplaat.hoogte_m,
        hoogte_zijplaat,
        concept.dikte_achterplaat_m,
        F_panel,
    )

    notch_area = hoogte_inkeping * lengte_inkeping
    zijplaat_B_mass_corr = (
        max(lengte_zijplaatB * hoogte_zijplaat - notch_area, 0.0)
        * concept.dikte_zijplaat_m
        * MATERIALEN[cfg_b.voorkeursmateriaal_behuizing].rho_kg_m3
    )
    massa_panelen = (
        2.0 * paneel_A["massa_kg"]
        + 2.0 * zijplaat_B_mass_corr
        + voorplaat["massa_kg"]
        + achterplaat["massa_kg"]
    )
    massa_support_totaal = 2.0 * props_support["massa_totaal_kg"]
    massa_totaal = massa_panelen + massa_support_totaal

    sigma_panel_max = np.nanmax([
        paneel_A["sigma_Pa"],
        paneel_B["sigma_Pa"],
        voorplaat["sigma_Pa"],
        achterplaat["sigma_Pa"],
    ])
    delta_panel_max = np.nanmax([
        paneel_A["doorbuiging_mm"],
        paneel_B["doorbuiging_mm"],
        voorplaat["doorbuiging_mm"],
        achterplaat["doorbuiging_mm"],
    ])
    sigma_panel_toeg = MATERIALEN[cfg_b.voorkeursmateriaal_behuizing].sigma_vloei_Pa / cfg_b.veiligheidsfactor_behuizing

    if sigma_support > sigma_support_toeg:
        haalbaar = False
        redenen.append("ondersteuningsplaat spanning te hoog")
    if 1000.0 * delta_support > cfg_b.toegelaten_doorbuiging_ondersteuningsplaat_mm:
        haalbaar = False
        redenen.append("ondersteuningsplaat doorbuiging te hoog")
    if theta_support > cfg_b.toegelaten_rotatie_ondersteuningsplaat_graden:
        haalbaar = False
        redenen.append("ondersteuningsplaat rotatie te hoog")
    if sigma_panel_max > sigma_panel_toeg:
        haalbaar = False
        redenen.append("paneelspanning te hoog")
    if delta_panel_max > cfg_b.toegelaten_doorbuiging_paneel_mm:
        haalbaar = False
        redenen.append("paneeldoorbuiging te hoog")
    if massa_totaal > cfg_b.toegelaten_massa_behuizing_kg:
        haalbaar = False
        redenen.append("behuizingsmassa te hoog")

    doel = (
        veilige_del(sigma_support, sigma_support_toeg, float("nan")) * 4000.0
        + veilige_del(1000.0 * delta_support, cfg_b.toegelaten_doorbuiging_ondersteuningsplaat_mm, float("nan")) * 4000.0
        + veilige_del(theta_support, cfg_b.toegelaten_rotatie_ondersteuningsplaat_graden, float("nan")) * 3000.0
        + veilige_del(sigma_panel_max, sigma_panel_toeg, float("nan")) * 2500.0
        + veilige_del(delta_panel_max, cfg_b.toegelaten_doorbuiging_paneel_mm, float("nan")) * 2500.0
        + veilige_del(massa_totaal, cfg_b.toegelaten_massa_behuizing_kg, float("nan"))
        + (0.0 if haalbaar else 1e6)
    )

    return {
        "haalbaar": haalbaar,
        "redenen": redenen,
        "doelfunctie": doel,
        "materiaal": concept.ondersteuningsplaat.materiaal,
        "lengte_ondersteuningsplaat_mm": mm(concept.ondersteuningsplaat.breedte_m),
        "breedte_ondersteuningsplaat_mm": mm(concept.ondersteuningsplaat.hoogte_m),
        "dikte_ondersteuningsplaat_mm": mm(concept.ondersteuningsplaat.dikte_m),
        "aantal_ribben": concept.ondersteuningsplaat.aantal_ribben,
        "hoogte_rib_mm": mm(concept.ondersteuningsplaat.hoogte_rib_m),
        "dikte_rib_mm": mm(concept.ondersteuningsplaat.dikte_rib_m),
        "dikte_zijplaat_mm": mm(concept.dikte_zijplaat_m),
        "dikte_voorplaat_mm": mm(concept.dikte_voorplaat_m),
        "dikte_achterplaat_mm": mm(concept.dikte_achterplaat_m),
        "hoogte_zijplaat_mm": mm(hoogte_zijplaat),
        "lengte_zijplaatA_mm": mm(lengte_zijplaatA),
        "lengte_zijplaatB_mm": mm(lengte_zijplaatB),
        "hoogte_inkeping_mm": mm(hoogte_inkeping),
        "lengte_inkeping_mm": mm(lengte_inkeping),
        "hoogte_voorplaat_mm": mm(hoogte_voorplaat),
        "kracht_behuizing_N": F_behuizing,
        "verdeling_riemkracht_as": verdeling_riemkracht_as,
        "k_plaat_single_N_m": k_plaat_single,
        "k_behuizing_N_m": k_behuizing,
        "sigma_support_Pa": sigma_support,
        "sigma_support_toeg_Pa": sigma_support_toeg,
        "doorbuiging_support_mm": 1000.0 * delta_support,
        "rotatie_support_graden": theta_support,
        "sigma_panel_max_Pa": sigma_panel_max,
        "sigma_panel_toeg_Pa": sigma_panel_toeg,
        "doorbuiging_panel_max_mm": delta_panel_max,
        "massa_support_totaal_kg": massa_support_totaal,
        "massa_panelen_kg": massa_panelen,
        "massa_totaal_kg": massa_totaal,
        "benutting_support_spanning": veilige_del(sigma_support, sigma_support_toeg, float("nan")),
        "benutting_support_doorbuiging": veilige_del(1000.0 * delta_support, cfg_b.toegelaten_doorbuiging_ondersteuningsplaat_mm, float("nan")),
        "benutting_support_rotatie": veilige_del(theta_support, cfg_b.toegelaten_rotatie_ondersteuningsplaat_graden, float("nan")),
        "benutting_panel_spanning": veilige_del(sigma_panel_max, sigma_panel_toeg, float("nan")),
        "benutting_panel_doorbuiging": veilige_del(delta_panel_max, cfg_b.toegelaten_doorbuiging_paneel_mm, float("nan")),
        "benutting_massa": veilige_del(massa_totaal, cfg_b.toegelaten_massa_behuizing_kg, float("nan")),
    }


def optimaliseer_behuizing(
    cfg_b: ConfiguratieBehuizing,
    interface_lasten: Dict[str, float],
    m2: Dict[str, float],
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    rijen = []
    for materiaal in [cfg_b.voorkeursmateriaal_behuizing]:
        for l_mm in cfg_b.breedtes_ondersteuningsplaat_mm:
            for b_mm in cfg_b.hoogtes_ondersteuningsplaat_mm:
                for t_mm in cfg_b.diktes_ondersteuningsplaat_mm:
                    for n_rib in cfg_b.aantallen_ribben:
                        for hr_mm in cfg_b.hoogtes_rib_mm:
                            for tr_mm in cfg_b.diktes_rib_mm:
                                if n_rib == 0 and (hr_mm != 0 or tr_mm != 0):
                                    continue
                                if n_rib > 0 and (hr_mm == 0 or tr_mm == 0):
                                    continue
                                for t_zij_mm in cfg_b.diktes_zijplaat_mm:
                                    for t_voor_mm in cfg_b.diktes_voorplaat_mm:
                                        for t_achter_mm in cfg_b.diktes_achterplaat_mm:
                                            concept = BehuizingConcept(
                                                ondersteuningsplaat=Plaatconcept(
                                                    materiaal,
                                                    m_uit_mm(l_mm),
                                                    m_uit_mm(b_mm),
                                                    m_uit_mm(t_mm),
                                                    n_rib,
                                                    m_uit_mm(hr_mm),
                                                    m_uit_mm(tr_mm),
                                                ),
                                                dikte_zijplaat_m=m_uit_mm(t_zij_mm),
                                                dikte_voorplaat_m=m_uit_mm(t_voor_mm),
                                                dikte_achterplaat_m=m_uit_mm(t_achter_mm),
                                            )
                                            rijen.append(evalueer_behuizingsconcept(concept, cfg_b, interface_lasten, m2))
    df = pd.DataFrame(rijen).sort_values(["haalbaar", "doelfunctie"], ascending=[False, True]).reset_index(drop=True)
    haalbaar = df[df["haalbaar"]].reset_index(drop=True)
    return df, haalbaar


def maak_voorkeursbehuizing(cfg_b: ConfiguratieBehuizing) -> BehuizingConcept:
    return BehuizingConcept(
        ondersteuningsplaat=Plaatconcept(
            cfg_b.voorkeursmateriaal_behuizing,
            m_uit_mm(cfg_b.voorkeurs_lengte_ondersteuningsplaat_mm),
            m_uit_mm(cfg_b.voorkeurs_breedte_ondersteuningsplaat_mm),
            m_uit_mm(cfg_b.voorkeurs_dikte_ondersteuningsplaat_mm),
            cfg_b.voorkeurs_aantal_ribben,
            m_uit_mm(cfg_b.voorkeurs_hoogte_rib_mm),
            m_uit_mm(cfg_b.voorkeurs_dikte_rib_mm),
        ),
        dikte_zijplaat_m=m_uit_mm(cfg_b.voorkeurs_dikte_zijplaat_mm),
        dikte_voorplaat_m=m_uit_mm(cfg_b.voorkeurs_dikte_voorplaat_mm),
        dikte_achterplaat_m=m_uit_mm(cfg_b.voorkeurs_dikte_achterplaat_mm),
    )


def maak_overzicht_voorkeursframe(frame_voorkeur: Dict[str, object]) -> pd.DataFrame:
    return pd.DataFrame([
        ["lengte frame", mm(cfg_frame.lengte_frame_m), "mm"],
        ["breedte frame", frame_voorkeur["breedte_mm"], "mm"],
        ["hoogte frame", frame_voorkeur["hoogte_mm"], "mm"],
        ["wanddikte", frame_voorkeur["wanddikte_mm"], "mm"],
        ["massa frame", frame_voorkeur["massa_kg"], "kg"],
        ["doorbuiging frame", frame_voorkeur["doorbuiging_mm"], "mm"],
        ["rotatie frame", frame_voorkeur["rotatie_deg"], "deg"],
        ["spanning frame", frame_voorkeur["sigma_eq_Pa"] / 1e6, "MPa"],
        ["haalbaar", frame_voorkeur["haalbaar"], "-"],
    ], columns=["grootheid", "waarde", "eenheid"])


def maak_overzicht_voorkeursbehuizing(behuizing_voorkeur: Dict[str, object]) -> pd.DataFrame:
    return pd.DataFrame([
        ["lengte ondersteuningsplaat", behuizing_voorkeur["lengte_ondersteuningsplaat_mm"], "mm"],
        ["breedte ondersteuningsplaat", behuizing_voorkeur["breedte_ondersteuningsplaat_mm"], "mm"],
        ["dikte ondersteuningsplaat", behuizing_voorkeur["dikte_ondersteuningsplaat_mm"], "mm"],
        ["hoogte zijplaat", behuizing_voorkeur["hoogte_zijplaat_mm"], "mm"],
        ["dikte zijplaat", behuizing_voorkeur["dikte_zijplaat_mm"], "mm"],
        ["hoogte inkeping", behuizing_voorkeur["hoogte_inkeping_mm"], "mm"],
        ["lengte inkeping", behuizing_voorkeur["lengte_inkeping_mm"], "mm"],
        ["kracht naar behuizing", behuizing_voorkeur["kracht_behuizing_N"], "N"],
        ["doorbuiging ondersteuningsplaat", behuizing_voorkeur["doorbuiging_support_mm"], "mm"],
        ["doorbuiging panelen max", behuizing_voorkeur["doorbuiging_panel_max_mm"], "mm"],
        ["massa totaal", behuizing_voorkeur["massa_totaal_kg"], "kg"],
        ["haalbaar", behuizing_voorkeur["haalbaar"], "-"],
    ], columns=["grootheid", "waarde", "eenheid"])


def lokale_studie_voorkeursframe(cfg: ConfiguratieFrame, interface_lasten: Dict[str, float], concept: FrameConcept) -> pd.DataFrame:
    rijen = []
    for t_mm in cfg.wanddiktes_frame_mm:
        test = FrameConcept(concept.materiaal, concept.breedte_buiten_m, concept.hoogte_buiten_m, m_uit_mm(t_mm))
        rijen.append(evalueer_frameconcept(test, cfg, interface_lasten))
    return pd.DataFrame(rijen).sort_values("doelfunctie").reset_index(drop=True)


def lokale_studie_voorkeursbehuizing(
    cfg_b: ConfiguratieBehuizing,
    interface_lasten: Dict[str, float],
    m2: Dict[str, float],
    concept: BehuizingConcept,
) -> pd.DataFrame:
    rijen = []
    for t_mm in cfg_b.diktes_ondersteuningsplaat_mm:
        for tz_mm in cfg_b.diktes_zijplaat_mm:
            test = BehuizingConcept(
                ondersteuningsplaat=Plaatconcept(
                    concept.ondersteuningsplaat.materiaal,
                    concept.ondersteuningsplaat.breedte_m,
                    concept.ondersteuningsplaat.hoogte_m,
                    m_uit_mm(t_mm),
                    concept.ondersteuningsplaat.aantal_ribben,
                    concept.ondersteuningsplaat.hoogte_rib_m,
                    concept.ondersteuningsplaat.dikte_rib_m,
                ),
                dikte_zijplaat_m=m_uit_mm(tz_mm),
                dikte_voorplaat_m=concept.dikte_voorplaat_m,
                dikte_achterplaat_m=concept.dikte_achterplaat_m,
            )
            rijen.append(evalueer_behuizingsconcept(test, cfg_b, interface_lasten, m2))
    return pd.DataFrame(rijen).sort_values("doelfunctie").reset_index(drop=True)


def plot_tradeoff_frame(df: pd.DataFrame) -> None:
    plt.figure(figsize=(7, 4))
    plt.scatter(df["massa_kg"], df["doorbuiging_mm"])
    plt.xlabel("massa [kg]")
    plt.ylabel("doorbuiging [mm]")
    plt.title("Trade-off frame")
    plt.grid(True)
    plt.tight_layout()
    plt.show()



def plot_tradeoff_behuizing(df: pd.DataFrame) -> None:
    plt.figure(figsize=(7, 4))
    plt.scatter(df["massa_totaal_kg"], df["doorbuiging_support_mm"])
    plt.xlabel("massa [kg]")
    plt.ylabel("doorbuiging ondersteuningsplaat [mm]")
    plt.title("Trade-off M2-behuizing")
    plt.grid(True)
    plt.tight_layout()
    plt.show()



def plot_layout_voorkeursbehuizing(behuizing_voorkeur: Dict[str, object]) -> None:
    fig, ax = plt.subplots(figsize=(8, 5))
    L = m_uit_mm(behuizing_voorkeur["lengte_ondersteuningsplaat_mm"])
    B = m_uit_mm(behuizing_voorkeur["breedte_ondersteuningsplaat_mm"])
    H = m_uit_mm(behuizing_voorkeur["hoogte_zijplaat_mm"])
    ink_h = m_uit_mm(behuizing_voorkeur["hoogte_inkeping_mm"])
    ink_l = m_uit_mm(behuizing_voorkeur["lengte_inkeping_mm"])

    ax.add_patch(plt.Rectangle((0.0, 0.0), L, H, fill=False, linewidth=2.0, label="zijplaat buitencontour"))
    ax.add_patch(plt.Rectangle((L - ink_l, 0.5 * (H - ink_h)), ink_l, ink_h, alpha=0.25, label="arm-inkeping"))
    ax.add_patch(plt.Rectangle((0.0, 0.0), L, B, fill=False, linestyle="--", label="ondersteuningsplaat planvorm"))
    ax.set_aspect("equal")
    ax.set_xlabel("x [m]")
    ax.set_ylabel("z [m]")
    ax.set_title("Voorkeursconcept M2-behuizing")
    ax.grid(True)
    ax.legend()
    plt.tight_layout()
    plt.show()


cfg_m2 = maak_m2_config()
m2_kengetallen = bereken_m2_kengetallen(cfg_m2)
interface_lasten = maak_interface_lasten_frame()
cfg_frame = maak_afgeleide_frameconfig()
cfg_behuizing = maak_afgeleide_behuizingconfig(cfg_m2, m2_kengetallen)

frame_alle, frame_haalbaar = optimaliseer_frame(cfg_frame, interface_lasten)
voorkeursframe = maak_voorkeursframe(cfg_frame)
frame_voorkeur = evalueer_frameconcept(voorkeursframe, cfg_frame, interface_lasten)
frame_lokale_studie = lokale_studie_voorkeursframe(cfg_frame, interface_lasten, voorkeursframe)
overzicht_voorkeursframe = maak_overzicht_voorkeursframe(frame_voorkeur)

behuizing_alle, behuizing_haalbaar = optimaliseer_behuizing(cfg_behuizing, interface_lasten, m2_kengetallen)
voorkeursbehuizing = maak_voorkeursbehuizing(cfg_behuizing)
behuizing_voorkeur = evalueer_behuizingsconcept(voorkeursbehuizing, cfg_behuizing, interface_lasten, m2_kengetallen)
behuizing_lokale_studie = lokale_studie_voorkeursbehuizing(cfg_behuizing, interface_lasten, m2_kengetallen, voorkeursbehuizing)
overzicht_voorkeursbehuizing = maak_overzicht_voorkeursbehuizing(behuizing_voorkeur)

print("\n=== M2 BASISGEGEVENS ===")
print(pd.Series(m2_kengetallen).to_string())
print("\n=== INTERFACE LASTEN FRAME / BEHUIZING ===")
print(pd.Series(interface_lasten).to_string())

if len(frame_haalbaar) > 0:
    print("\n=== BESTE HAALBARE FRAMECONCEPTEN ===")
    toon_dataframe("top frameconcepten", frame_haalbaar.head(15))
print("\n=== VALIDATIE VOORKEURSFRAME ===")
print(pd.Series(frame_voorkeur).to_string())
toon_dataframe("overzicht voorkeursframe", overzicht_voorkeursframe)
toon_dataframe("lokale studie voorkeursframe", frame_lokale_studie.head(20))

if len(behuizing_haalbaar) > 0:
    print("\n=== BESTE HAALBARE BEHUIZINGSCONCEPTEN ===")
    toon_dataframe("top behuizingsconcepten", behuizing_haalbaar.head(15))
print("\n=== VALIDATIE VOORKEURSBEHUIZING ===")
print(pd.Series(behuizing_voorkeur).to_string())
toon_dataframe("overzicht voorkeursbehuizing", overzicht_voorkeursbehuizing)
toon_dataframe("lokale studie voorkeursbehuizing", behuizing_lokale_studie.head(20))

resultaten_frame_behuizing = {
    "m2_configuratie": asdict(cfg_m2),
    "m2_kengetallen": m2_kengetallen.copy(),
    "configuratie_frame": asdict(cfg_frame),
    "configuratie_behuizing": asdict(cfg_behuizing),
    "interface_lasten": interface_lasten.copy(),
    "frame_alle": frame_alle.copy(),
    "frame_haalbaar": frame_haalbaar.copy(),
    "frame_voorkeur": frame_voorkeur.copy(),
    "frame_lokale_studie": frame_lokale_studie.copy(),
    "behuizing_alle": behuizing_alle.copy(),
    "behuizing_haalbaar": behuizing_haalbaar.copy(),
    "behuizing_voorkeur": behuizing_voorkeur.copy(),
    "behuizing_lokale_studie": behuizing_lokale_studie.copy(),
    "overzicht_voorkeursframe": overzicht_voorkeursframe.copy(),
    "overzicht_voorkeursbehuizing": overzicht_voorkeursbehuizing.copy(),
    "export_voor_volgende_modules": {
        "frame_voorkeur_haalbaar": bool(frame_voorkeur["haalbaar"]),
        "frame_lengte_mm": float(mm(cfg_frame.lengte_frame_m)),
        "frame_breedte_mm": float(frame_voorkeur["breedte_mm"]),
        "frame_hoogte_mm": float(frame_voorkeur["hoogte_mm"]),
        "frame_wanddikte_mm": float(frame_voorkeur["wanddikte_mm"]),
        "frame_massa_kg": float(frame_voorkeur["massa_kg"]),
        "frame_doorbuiging_mm": float(frame_voorkeur["doorbuiging_mm"]),
        "frame_rotatie_deg": float(frame_voorkeur["rotatie_deg"]),
        "frame_sigma_eq_MPa": float(frame_voorkeur["sigma_eq_Pa"] / 1e6),
        "behuizing_voorkeur_haalbaar": bool(behuizing_voorkeur["haalbaar"]),
        "m2_module_lengte_mm": float(behuizing_voorkeur["lengte_ondersteuningsplaat_mm"]),
        "m2_module_breedte_mm": float(behuizing_voorkeur["breedte_ondersteuningsplaat_mm"]),
        "m2_module_hoogte_mm": float(behuizing_voorkeur["hoogte_zijplaat_mm"]),
        "m2_ondersteuningsplaat_dikte_mm": float(behuizing_voorkeur["dikte_ondersteuningsplaat_mm"]),
        "m2_zijplaat_dikte_mm": float(behuizing_voorkeur["dikte_zijplaat_mm"]),
        "m2_voorplaat_dikte_mm": float(behuizing_voorkeur["dikte_voorplaat_mm"]),
        "m2_achterplaat_dikte_mm": float(behuizing_voorkeur["dikte_achterplaat_mm"]),
        "m2_inkeping_hoogte_mm": float(behuizing_voorkeur["hoogte_inkeping_mm"]),
        "m2_inkeping_lengte_mm": float(behuizing_voorkeur["lengte_inkeping_mm"]),
        "m2_behuizing_kracht_N": float(behuizing_voorkeur["kracht_behuizing_N"]),
        "m2_support_doorbuiging_mm": float(behuizing_voorkeur["doorbuiging_support_mm"]),
        "m2_panel_doorbuiging_max_mm": float(behuizing_voorkeur["doorbuiging_panel_max_mm"]),
        "m2_behuizing_massa_kg": float(behuizing_voorkeur["massa_totaal_kg"]),
        "behuizing_lengte_mm": float(mm(cfg_frame.lengte_frame_m)),
        "behuizing_breedte_mm": float(frame_voorkeur["breedte_mm"]),
        "behuizing_diepte_mm": float(frame_voorkeur["hoogte_mm"]),
        "behuizing_sigma_eq_MPa": float(frame_voorkeur["sigma_eq_Pa"] / 1e6),
        "behuizing_delta_max_mm": float(frame_voorkeur["doorbuiging_mm"]),
        "behuizing_ok_spanning": bool(frame_voorkeur["haalbaar"] and frame_voorkeur["sigma_eq_Pa"] <= frame_voorkeur["sigma_toeg_Pa"]),
        "behuizing_ok_deflectie": bool(frame_voorkeur["doorbuiging_mm"] <= cfg_frame.toegelaten_doorbuiging_frame_mm),
        "behuizing_ok_rotatie": bool(frame_voorkeur["rotatie_deg"] <= cfg_frame.toegelaten_rotatie_frame_deg),
    },
}

print(resultaten_frame_behuizing["export_voor_volgende_modules"])



=== M2 BASISGEGEVENS ===
alpha_max_rad_s2                         3.534292
alpha_gem_rad_s2                         0.785398
omega_max_rad_s                          2.356194
omega_gem_rad_s                          1.570796
massa_uitgaandearm_kg                    9.088200
I_uitgaandearm_m4                        0.000006
J_totaal_kgm2                            2.176292
T_dynamisch_Nm                           7.691652
M_statisch_Nm                           47.690981
M_totaal_Nm                             48.307258
sigma_arm_Pa                        381423.280234
P_motor_W                               18.123027
n_uitgaandeas_rpm                       22.500000
lengte_uitgaandeas_m                     0.600000
M_max_doorbuiging_as_Nm                182.024860
T_tandriem_Nm                           11.537478
steekcirkeldiameter_tandriem_m           0.238732
definitieve_lengte_tandriem_m            1.075000
definitieve_asafstand_tandriem_m         0.162500
snelheid_tandriem_m_s   